In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

# Update folders to match your directory setup
IMAGES_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\Ish\images"
MASKS_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\Ish\refined_masks"
OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\Ish\scale_calibration.csv"

def extract_scale_factor_per_image(image_path, physical_length_nm=1000.0):
    """
    Dynamically measures the scale bar length for an individual TEM image.
    Returns: (resolution_nm_per_px, scale_bar_pixels)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None, None

    h, w = img.shape
    # Crop the scale bar region in the lower-right corner (bottom 15%, right 30%)
    crop = img[int(h * 0.85):, int(w * 0.70):]
    
    # Threshold to isolate the dark scale line inside the light container box
    _, binary = cv2.threshold(crop, 50, 255, cv2.THRESH_BINARY_INV)
    
    # Horizontal projection to measure line length
    vertical_projection = np.sum(binary == 255, axis=0)
    line_indices = np.where(vertical_projection > 0)[0]
    
    if len(line_indices) >= 2:
        scale_bar_px = line_indices[-1] - line_indices[0] + 1
        nm_per_pixel = physical_length_nm / scale_bar_px
        return nm_per_pixel, scale_bar_px
    else:
        return None, None

# Run dynamic batch scale extraction
scale_dict = {}
records = []

if os.path.exists(IMAGES_FOLDER):
    print("=" * 65)
    print(" BATCH EXTRACTING DYNAMIC SCALES PER IMAGE ")
    print("=" * 65)
    
    for filename in sorted(os.listdir(IMAGES_FOLDER)):
        if filename.endswith(('.tif', '.png', '.jpg')):
            img_path = os.path.join(IMAGES_FOLDER, filename)
            res_nm_px, bar_px = extract_scale_factor_per_image(img_path)
            
            # Key for mapping (using base filename without extension)
            base_key = os.path.splitext(filename)[0]
            
            if res_nm_px is not None:
                scale_dict[base_key] = res_nm_px
                records.append({
                    "image_name": filename,
                    "scale_bar_px": bar_px,
                    "nm_per_pixel": round(res_nm_px, 4)
                })
                print(f"✓ {filename:<20} | Scale Bar: {bar_px:>3} px | Res: {res_nm_px:.4f} nm/px")
            else:
                print(f"✗ {filename:<20} | Could not detect scale bar!")

    # Save scale table to CSV for reference
    df_scales = pd.DataFrame(records)
    df_scales.to_csv(OUTPUT_CSV, index=False)
    print("=" * 65)
    print(f"Saved exact calibrations for {len(records)} images to '{OUTPUT_CSV}'.")
    print("=" * 65)
else:
    print(f"[Error] Directory not found: '{IMAGES_FOLDER}'")